# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.
- Load nyc taxi data from January of 2025 and compare data.
- With any remaining time, work on the where to go from here section.

---

**Choice for this notebook:** the main lab (Parts 1 and 2) is done with **PySpark (DataFrame API)**, Part 3 is redone in **Spark SQL**.

In [1]:
import os
import requests

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# start a spark session and create a spark context
# (driver memory limited to 2 GB because Docker Desktop only has ~4 GB of RAM)
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext

In [2]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = "https://d37ci6vzurychx.cloudfront.net"

def download(url, path):
    # download url to path, unless the file is already there
    if not os.path.exists(path):
        response = requests.get(url)
        response.raise_for_status()
        with open(path, "wb") as f:
            f.write(response.content)
    return path

def download_trips(year, month):
    name = f"yellow_tripdata_{year}-{month:02d}.parquet"
    return download(f"{BASE_URL}/trip-data/{name}", os.path.join(DATA_DIR, name))

# get the January 2019 trip data
jan_2019_trip_data = download_trips(2019, 1)

In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show(5)
print("Number of rows:", df_trips.count())

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

Number of rows: 7696617


## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

#### 1.1 Unique key

`F.monotonically_increasing_id()` generates a **unique** (but not consecutive) id for each row, without moving all the data to a single partition (unlike `row_number()` over a window without `partitionBy`).

The time-related columns needed by the next questions (duration, date, hour, day of week, time of day) are added at the same time. Everything is wrapped in a function so it can be reused on the January 2025 data in Part 2.

In [5]:
# time of day buckets: (label, start hour, end hour excluded)
TIME_OF_DAY = [("late night (0h-6h)", 0, 6), ("morning (6h-12h)", 6, 12),
               ("afternoon (12h-17h)", 12, 17), ("evening (17h-24h)", 17, 24)]
TOD_LABELS = [label for label, _, _ in TIME_OF_DAY]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def time_of_day(hour_col):
    expr = None
    for label, start, end in TIME_OF_DAY:
        cond = (hour_col >= start) & (hour_col < end)
        expr = F.when(cond, label) if expr is None else expr.when(cond, label)
    return expr

def enrich(df):
    return (df
        .withColumn("trip_id", F.monotonically_increasing_id())
        .withColumn("duration_min",
                    F.expr("timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime)") / 60)
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("day_num", F.expr("weekday(tpep_pickup_datetime)") + 1)   # 1 = Monday ... 7 = Sunday
        .withColumn("day_name", F.date_format("tpep_pickup_datetime", "EEEE"))
        .withColumn("time_of_day", time_of_day(F.col("pickup_hour"))))

df_trips = enrich(df_trips)
df_trips.select("trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_min",
                "pickup_date", "pickup_hour", "day_name", "time_of_day").show(5)

+-----------+--------------------+---------------------+------------------+-----------+-----------+---------+-------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|      duration_min|pickup_date|pickup_hour| day_name|        time_of_day|
+-----------+--------------------+---------------------+------------------+-----------+-----------+---------+-------------------+
|25769803776| 2019-01-01 00:46:40|  2019-01-01 00:53:20| 6.666666666666667| 2019-01-01|          0|  Tuesday| late night (0h-6h)|
|25769803777| 2019-01-01 00:59:47|  2019-01-01 01:18:59|              19.2| 2019-01-01|          0|  Tuesday| late night (0h-6h)|
|25769803778| 2018-12-21 13:48:30|  2018-12-21 13:52:40| 4.166666666666667| 2018-12-21|         13|   Friday|afternoon (12h-17h)|
|25769803779| 2018-11-28 15:52:25|  2018-11-28 15:55:45|3.3333333333333335| 2018-11-28|         15|Wednesday|afternoon (12h-17h)|
|25769803780| 2018-11-28 15:56:57|  2018-11-28 15:58:33|               1.6| 2018-11-28|   

#### Raw vs cleaned data

The last question of Part 1 shows that the dataset contains outliers (dates outside January 2019, negative durations, negative amounts...). So that averages and rankings make sense, two views of the data are defined now:

- `df_jan`: every trip picked up **in January 2019** (used for counts: busiest days / hours);
- `df_clean`: `df_jan` restricted to plausible values (used for averages, tips, etc.):
  - duration between 1 min and 3 h,
  - distance between 0.1 and 100 miles,
  - 1 to 6 passengers (or missing: ~0.4% of trips in 2019 but ~15% in 2025, dropping them would bias the comparison),
  - fare between 0 (excluded) and 500 $, total > 0.

The "highest / lowest" questions are answered on the **raw data** and then on the **cleaned data**.

In [6]:
def january_only(df, year):
    return df.filter(F.col("pickup_date").between(f"{year}-01-01", f"{year}-01-31"))

def clean(df):
    return df.filter(
        F.col("duration_min").between(1, 180)
        & F.col("trip_distance").between(0.1, 100)
        & (F.col("passenger_count").isNull() | F.col("passenger_count").between(1, 6))
        & (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500)
        & (F.col("total_amount") > 0))

df_jan = january_only(df_trips, 2019)
df_clean = clean(df_jan)

n_raw, n_jan, n_clean = df_trips.count(), df_jan.count(), df_clean.count()
print(f"raw      : {n_raw:>10,}")
print(f"january  : {n_jan:>10,}  ({n_jan / n_raw:.2%})")
print(f"cleaned  : {n_clean:>10,}  ({n_clean / n_raw:.2%})")

raw      :  7,696,617
january  :  7,696,080  (99.99%)
cleaned  :  7,465,335  (97.00%)


#### 1.2 Which trip has the highest passenger count?

In [7]:
max_passengers = df_trips.agg(F.max("passenger_count")).first()[0]
top_passengers = df_trips.filter(F.col("passenger_count") == max_passengers)
print(f"Highest passenger count: {max_passengers:.0f}")
print(f"Number of trips with {max_passengers:.0f} passengers: {top_passengers.count()}")

top_passengers.select("trip_id", "tpep_pickup_datetime", "passenger_count", "trip_distance",
                      "duration_min", "fare_amount", "total_amount") \
    .orderBy(F.desc("trip_distance")).show(10)

Highest passenger count: 9


Number of trips with 9 passengers: 9


+-----------+--------------------+---------------+-------------+-------------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|passenger_count|trip_distance|       duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------+-------------+-------------------+-----------+------------+
|25777177652| 2019-01-30 22:17:51|            9.0|        13.38| 27.033333333333335|       90.0|        90.8|
|25770753732| 2019-01-05 13:12:29|            9.0|          0.0|               0.05|        9.8|        12.6|
|25771100063| 2019-01-07 03:19:36|            9.0|          0.0| 0.4166666666666667|        9.0|         9.3|
|25771815874| 2019-01-10 00:43:10|            9.0|          0.0|0.06666666666666667|        9.0|        11.3|
|25772687771| 2019-01-13 04:13:24|            9.0|          0.0| 1.1666666666666667|        9.0|       12.25|
|25774338483| 2019-01-19 16:45:25|            9.0|          0.0|0.03333333333333333|       92.0|      110.76|
|257746560

**Answer:** the highest passenger count is **9**, for **9 trips**. However 8 of them have a distance of 0 miles and last less than 1.2 minutes: they are data entry errors (a yellow cab can legally carry 4 to 5 passengers). The only "credible" one is `trip_id = 25777177652` (Jan 30, 2019 at 22:17, 13.38 miles, 27 min, 90 $), probably a large van.

#### 1.3 Average passenger count

In [8]:
df_trips.agg(F.round(F.avg("passenger_count"), 3).alias("avg_passengers")).show()

+--------------+
|avg_passengers|
+--------------+
|         1.567|
+--------------+



**Answer:** on average **1.57 passengers per trip**. Note that `avg` ignores the 28,672 null values, and that 117,381 trips have 0 passengers (value not entered by the driver).

#### 1.4 Shortest / longest trips — by distance, then by duration

In [9]:
trip_cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
             "duration_min", "fare_amount", "total_amount"]

print("=== Longest by distance (raw) ===")
df_trips.orderBy(F.desc("trip_distance")).select(trip_cols).show(3)
print("Trips with distance = 0:", df_trips.filter(F.col("trip_distance") == 0).count())

print("=== Longest by distance (cleaned) ===")
df_clean.orderBy(F.desc("trip_distance")).select(trip_cols).show(3)
print("=== Shortest by distance (cleaned) ===")
df_clean.orderBy("trip_distance", "duration_min").select(trip_cols).show(3)

=== Longest by distance (raw) ===


+-----------+--------------------+---------------------+-------------+-----------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|     duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+-----------------+-----------+------------+
|25775877867| 2019-01-25 21:56:39|  2019-01-25 22:06:08|        831.8|9.483333333333333|        8.5|       11.76|
|25774090409| 2019-01-18 16:32:24|  2019-01-18 16:39:20|        700.7|6.933333333333334|        6.0|         9.0|
|25776574761| 2019-01-28 17:24:11|  2019-01-29 04:37:16|       214.01|673.0833333333334|      760.0|       761.8|
+-----------+--------------------+---------------------+-------------+-----------------+-----------+------------+
only showing top 3 rows


Trips with distance = 0: 55089
=== Longest by distance (cleaned) ===


+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|25772805306| 2019-01-13 17:40:12|  2019-01-13 20:02:32|        98.38|142.33333333333334|      300.0|       300.8|
|25774466892| 2019-01-20 03:44:56|  2019-01-20 05:18:54|        96.13| 93.96666666666667|      200.0|      276.96|
|25774027367| 2019-01-18 11:34:58|  2019-01-18 13:10:34|        91.86|              95.6|      240.0|       290.0|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
only showing top 3 rows
=== Shortest by distance (cleaned) ===


+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|25770552709| 2019-01-04 16:22:18|  2019-01-04 16:23:18|          0.1|         1.0|        2.5|         4.3|
|25771037988| 2019-01-06 17:56:45|  2019-01-06 17:57:45|          0.1|         1.0|        2.5|         3.3|
|25770620593| 2019-01-04 21:01:59|  2019-01-04 21:02:59|          0.1|         1.0|        3.0|         4.3|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
only showing top 3 rows


In [10]:
print("=== Longest by duration (raw) ===")
df_trips.orderBy(F.desc("duration_min")).select(trip_cols).show(3)
print("=== Shortest by duration (raw) ===")
df_trips.orderBy("duration_min").select(trip_cols).show(3)

print("=== Longest by duration (cleaned) ===")
df_clean.orderBy(F.desc("duration_min")).select(trip_cols).show(3)
print("=== Shortest by duration (cleaned) ===")
df_clean.orderBy("duration_min").select(trip_cols).show(3)

=== Longest by duration (raw) ===


+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|25769872043| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2| 43648.01666666667|        6.5|         7.3|
|25770396038| 2019-01-03 22:24:36|  2019-01-27 10:41:17|          1.1|33856.683333333334|        8.5|       11.15|
|25770679632| 2019-01-05 04:21:40|  2019-01-27 01:53:46|          3.9|           31532.1|       16.5|       21.35|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
only showing top 3 rows
=== Shortest by duration (raw) ===


+-----------+--------------------+---------------------+-------------+-------------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|       duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+-------------------+-----------+------------+
|25771006960| 2019-01-06 15:15:08|  2018-11-09 02:34:38|          3.3|           -84280.5|       16.0|        16.8|
|25772780069| 2019-01-13 15:15:27|  2018-12-25 07:37:43|          7.6|-27817.733333333334|       29.5|       36.06|
|25774545522| 2019-01-20 15:15:30|  2019-01-18 15:57:16|          1.9| -2838.233333333333|       15.5|       19.55|
+-----------+--------------------+---------------------+-------------+-------------------+-----------+------------+
only showing top 3 rows
=== Longest by duration (cleaned) ===


+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
|25772058424| 2019-01-10 21:00:00|  2019-01-11 00:00:00|         3.12|             180.0|       14.0|        17.6|
|25772335394| 2019-01-11 21:00:28|  2019-01-12 00:00:00|         1.34|179.53333333333333|        9.5|       12.96|
|25770262840| 2019-01-03 11:56:30|  2019-01-03 14:55:56|         4.43|179.43333333333334|       52.0|        52.8|
+-----------+--------------------+---------------------+-------------+------------------+-----------+------------+
only showing top 3 rows
=== Shortest by duration (cleaned) ===


+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|25769822029| 2019-01-01 01:54:11|  2019-01-01 01:55:11|         0.18|         1.0|        3.0|         4.3|
|25769819952| 2019-01-01 01:43:30|  2019-01-01 01:44:30|         0.22|         1.0|        3.0|         4.3|
|25769829475| 2019-01-01 01:28:46|  2019-01-01 01:29:46|          0.3|         1.0|        3.0|         4.3|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
only showing top 3 rows


**Answer:**

| | Raw data | Cleaned data |
|---|---|---|
| **Longest (distance)** | 831.8 miles in 9.5 min for 8.50 $ → impossible (> 5,000 mph), meter error | **98.38 miles** in 2h22 for 300 $ (trip out of the city) |
| **Shortest (distance)** | 0 miles (**55,089 trips**) | 0.1 mile (filter threshold) |
| **Longest (duration)** | 43,648 min (≈ 30 days) for 1.2 miles and 6.50 $ → meter never stopped | 180 min (filter threshold) — several end exactly at 00:00:00, which suggests an automatic close at midnight |
| **Shortest (duration)** | −84,280 min (drop-off **before** pick-up) | 1 min (filter threshold) |

All the raw extremes are errors; on the cleaned data the extremes sit on the filter thresholds.

#### 1.5 Busiest / slowest single day

In [11]:
daily = df_jan.groupBy("pickup_date", "day_num", "day_name").count()

print("Busiest days")
daily.orderBy(F.desc("count")).show(3)
print("Slowest days")
daily.orderBy("count").show(3)

Busiest days


+-----------+-------+--------+------+
|pickup_date|day_num|day_name| count|
+-----------+-------+--------+------+
| 2019-01-25|      5|  Friday|292499|
| 2019-01-11|      5|  Friday|291714|
| 2019-01-31|      4|Thursday|284625|
+-----------+-------+--------+------+
only showing top 3 rows
Slowest days


+-----------+-------+---------+------+
|pickup_date|day_num| day_name| count|
+-----------+-------+---------+------+
| 2019-01-01|      2|  Tuesday|189432|
| 2019-01-21|      1|   Monday|192826|
| 2019-01-02|      3|Wednesday|198737|
+-----------+-------+---------+------+
only showing top 3 rows


**Answer:**
- **Busiest day: Friday, January 25, 2019** (292,499 trips), just ahead of Friday, January 11 (291,714).
- **Slowest day: Tuesday, January 1, 2019** (189,432, New Year's Day), followed by **Monday, January 21** (192,826 — Martin Luther King Jr. Day, a public holiday, during a cold snap).

#### 1.6 Busiest / slowest time of day

The buckets do not have the same length, so the number of trips **per hour** in each bucket is also computed.

In [12]:
hourly = df_jan.groupBy("pickup_hour").count().orderBy("pickup_hour")
hourly.show(24)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|207758|
|          1|149242|
|          2|109413|
|          3| 78084|
|          4| 61423|
|          5| 75532|
|          6|178598|
|          7|304858|
|          8|373735|
|          9|365924|
|         10|361382|
|         11|375438|
|         12|401172|
|         13|404149|
|         14|433115|
|         15|452679|
|         16|420806|
|         17|468407|
|         18|515374|
|         19|475152|
|         20|423128|
|         21|409873|
|         22|369026|
|         23|281812|
+-----------+------+



In [13]:
df_jan.groupBy("time_of_day").agg(
        F.count("*").alias("trips"),
        F.countDistinct("pickup_hour").alias("nb_hours"),
        F.min("pickup_hour").alias("start_hour")) \
    .withColumn("trips_per_hour", F.round(F.col("trips") / F.col("nb_hours"))) \
    .orderBy("start_hour").drop("start_hour").show(truncate=False)

+-------------------+-------+--------+--------------+
|time_of_day        |trips  |nb_hours|trips_per_hour|
+-------------------+-------+--------+--------------+
|late night (0h-6h) |681452 |6       |113575.0      |
|morning (6h-12h)   |1959935|6       |326656.0      |
|afternoon (12h-17h)|2111921|5       |422384.0      |
|evening (17h-24h)  |2942772|7       |420396.0      |
+-------------------+-------+--------+--------------+



**Answer:**
- **Busiest hour: 18h** (515,374 trips, commute home); **slowest hour: 4h** (61,423, 8 times fewer).
- Demand rises sharply between 6h and 8h, stays high during the day and peaks between 17h and 19h.
- By bucket: the **evening** has the most trips overall (2.94 M), but per hour the **afternoon** (422k/h) and the evening (420k/h) are tied. **Late night** is by far the slowest (114k/h).

#### 1.7 On average, which day of the week is busiest / slowest?

January 2019 starts on a **Tuesday**: there are 5 Tuesdays, Wednesdays and Thursdays but only 4 of the other days, so the **average per day** must be used instead of a plain total.

In [14]:
dow = daily.groupBy("day_num", "day_name").agg(
        F.count("*").alias("nb_days"),
        F.round(F.avg("count")).alias("avg_trips_per_day")) \
    .orderBy("day_num")
dow.show()

+-------+---------+-------+-----------------+
|day_num| day_name|nb_days|avg_trips_per_day|
+-------+---------+-------+-----------------+
|      1|   Monday|      4|         226941.0|
|      2|  Tuesday|      5|         241815.0|
|      3|Wednesday|      5|         253046.0|
|      4| Thursday|      5|         271398.0|
|      5|   Friday|      4|         271788.0|
|      6| Saturday|      4|         252495.0|
|      7|   Sunday|      4|         214973.0|
+-------+---------+-------+-----------------+



**Answer:** on average **Friday** (271,788 trips/day) and **Thursday** (271,398) are the busiest days; **Sunday** is the slowest (214,973), followed by Monday (226,941, pulled down by the January 21 holiday). A plain total would have wrongly pointed to Thursday (5 Thursdays vs 4 Fridays).

#### 1.8 Does trip distance or number of passengers affect the tip amount?

Only **credit card** tips (`payment_type = 1`) are recorded — cash tips are always 0 — so the analysis is done on card payments only.

In [15]:
card = df_clean.filter(F.col("payment_type") == 1) \
    .withColumn("tip_pct", 100 * F.col("tip_amount") / F.col("fare_amount"))

distance_bucket = (F.when(F.col("trip_distance") < 1, "0-1 mi")
                    .when(F.col("trip_distance") < 2, "1-2 mi")
                    .when(F.col("trip_distance") < 5, "2-5 mi")
                    .when(F.col("trip_distance") < 10, "5-10 mi")
                    .when(F.col("trip_distance") < 20, "10-20 mi")
                    .otherwise("20+ mi"))

card.withColumn("distance_bucket", distance_bucket) \
    .groupBy("distance_bucket").agg(
        F.count("*").alias("trips"),
        F.min("trip_distance").alias("min_dist"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct")) \
    .orderBy("min_dist").drop("min_dist").show()

card.groupBy("passenger_count").agg(
        F.count("*").alias("trips"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")) \
    .orderBy("passenger_count").show()

+---------------+-------+-------+-----------+
|distance_bucket|  trips|avg_tip|avg_tip_pct|
+---------------+-------+-------+-----------+
|         0-1 mi|1339310|   1.32|       24.6|
|         1-2 mi|1863602|   1.81|       21.9|
|         2-5 mi|1404564|   2.66|       20.4|
|        5-10 mi| 425884|   4.68|       19.1|
|       10-20 mi| 284312|   8.11|       18.5|
|         20+ mi|  35900|  10.53|       17.2|
+---------------+-------+-------+-----------+



+---------------+-------+-------+
|passenger_count|  trips|avg_tip|
+---------------+-------+-------+
|            1.0|3900376|   2.51|
|            2.0| 774399|   2.59|
|            3.0| 216695|   2.56|
|            4.0|  91148|   2.57|
|            5.0| 229200|    2.6|
|            6.0| 141754|    2.6|
+---------------+-------+-------+



**Answer:**
- **Distance strongly affects the tip amount**: the average tip goes from 1.32 $ (< 1 mile) to 10.53 $ (> 20 miles), because the tip is proportional to the fare.
- As a **percentage** of the fare it is the opposite: 24.6% for trips under 1 mile vs 17.2% above 20 miles (suggested tip buttons and rounding up weigh more on small fares; long flat-rate airport trips are tipped less).
- **The number of passengers has no effect**: the average tip stays between 2.51 and 2.60 $ whatever the number of passengers.

#### 1.9 Highest `extra` charge and which trip

In 2019, `extra` should only be 0.50 $ (overnight, 20h-6h) or 1 $ (weekday rush hour, 16h-20h).

In [16]:
df_trips.orderBy(F.desc("extra")).select(
    "trip_id", "tpep_pickup_datetime", "trip_distance", "duration_min",
    "fare_amount", "extra", "total_amount").show(3)

+-----------+--------------------+-------------+-----------------+-----------+------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|     duration_min|fare_amount| extra|total_amount|
+-----------+--------------------+-------------+-----------------+-----------+------+------------+
|25775127259| 2019-01-23 08:58:09|          0.0|              0.0|  355676.98|535.38|   356214.78|
|25777257006| 2019-01-31 10:06:09|          0.0|              0.0|        4.5| 23.04|       28.34|
|25770346979| 2019-01-03 18:32:36|         16.6|72.88333333333334|       61.0|  18.5|        92.3|
+-----------+--------------------+-------------+-----------------+-----------+------+------------+
only showing top 3 rows


**Answer:** the highest `extra` is **535.38 $**, for trip `trip_id = 25775127259` (Jan 23, 2019 at 08:58). It is obviously an **error**: 0 miles, 0 seconds and a fare of 355,676.98 $. The second one (23.04 $) is also a 0-mile, 0-second trip; the realistic highest values are around 18.50 $ on long trips.

#### 1.10 Strange data points / outliers

Count the rows breaking common-sense rules (a row can fall into several categories).

In [17]:
speed_mph = F.col("trip_distance") / (F.col("duration_min") / 60)

checks = [
    ("pickup outside January 2019",       ~F.col("pickup_date").between("2019-01-01", "2019-01-31")),
    ("dropoff before pickup",             F.col("duration_min") < 0),
    ("duration = 0",                      F.col("duration_min") == 0),
    ("duration > 24 h",                   F.col("duration_min") > 24 * 60),
    ("distance = 0",                      F.col("trip_distance") == 0),
    ("distance > 100 miles",              F.col("trip_distance") > 100),
    ("average speed > 100 mph",           (F.col("duration_min") > 0) & (speed_mph > 100)),
    ("0 passengers",                      F.col("passenger_count") == 0),
    ("more than 6 passengers",            F.col("passenger_count") > 6),
    ("fare_amount < 0",                   F.col("fare_amount") < 0),
    ("fare_amount > 1000 $",              F.col("fare_amount") > 1000),
    ("tip > fare (fare > 0)",             (F.col("fare_amount") > 0) & (F.col("tip_amount") > F.col("fare_amount"))),
]

counts = df_trips.agg(*[F.sum(cond.cast("int")).alias(name) for name, cond in checks]).first()
spark.createDataFrame([(name, counts[name]) for name, _ in checks], ["check", "rows"]) \
    .withColumn("pct", F.round(100 * F.col("rows") / n_raw, 3)) \
    .show(truncate=False)

+---------------------------+------+-----+
|check                      |rows  |pct  |
+---------------------------+------+-----+
|pickup outside January 2019|537   |0.007|
|dropoff before pickup      |4     |0.0  |
|duration = 0               |6553  |0.085|
|duration > 24 h            |5     |0.0  |
|distance = 0               |55089 |0.716|
|distance > 100 miles       |32    |0.0  |
|average speed > 100 mph    |5723  |0.074|
|0 passengers               |117381|1.525|
|more than 6 passengers     |57    |0.001|
|fare_amount < 0            |7129  |0.093|
|fare_amount > 1000 $       |22    |0.0  |
|tip > fare (fare > 0)      |6455  |0.084|
+---------------------------+------+-----+



In [18]:
print("Pickups outside January 2019 (by month):")
df_trips.filter(~F.col("pickup_date").between("2019-01-01", "2019-01-31")) \
    .groupBy(F.date_format("pickup_date", "yyyy-MM").alias("month")).count().orderBy("month").show(20)

print("Fares > 1000 $:")
df_trips.filter(F.col("fare_amount") > 1000).select(trip_cols).orderBy(F.desc("fare_amount")).show(5)

Pickups outside January 2019 (by month):


+-------+-----+
|  month|count|
+-------+-----+
|2001-02|    1|
|2003-01|    2|
|2008-12|   22|
|2009-01|   50|
|2018-11|   11|
|2018-12|  355|
|2019-02|   72|
|2019-03|    5|
|2019-04|    6|
|2019-05|    1|
|2019-06|    2|
|2019-07|    6|
|2019-08|    1|
|2019-09|    1|
|2088-01|    2|
+-------+-----+

Fares > 1000 $:


+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|fare_amount|total_amount|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
|25772303431| 2019-01-11 19:33:15|  2019-01-11 19:53:09|          2.4|        19.9|  623259.86|   623261.66|
|25775127259| 2019-01-23 08:58:09|  2019-01-23 08:58:09|          0.0|         0.0|  355676.98|   356214.78|
|25771963747| 2019-01-10 16:08:10|  2019-01-10 16:08:10|          0.0|         0.0|    36090.3|     36090.3|
|25771696557| 2019-01-09 16:43:40|  2019-01-09 16:43:40|          0.0|         0.0|   34674.65|    34674.65|
|25771453227| 2019-01-08 16:09:44|  2019-01-08 16:09:44|          0.0|         0.0|   33023.53|    33023.53|
+-----------+--------------------+---------------------+-------------+------------+-----------+------------+
only showing top 5 

**Outlier analysis:**

1. **Dates outside the period (537 rows)**: pick-ups in 2001, 2003, 2008-2009 and even **2088** → badly set taximeter clocks (and a few late-December / early-February trips filed in the wrong month).
2. **Inconsistent durations**: 4 trips dropped off **before** being picked up, 6,553 trips of 0 seconds, 5 trips longer than 24 h (up to 30 days for a few dollars) → meter not started/stopped properly.
3. **Distances**: 55,089 trips of 0 miles (cancelled trips, GPS failure, flat fare entered without a ride) and 32 trips over 100 miles, including 831.8 miles in 9 minutes. **5,723 trips have an average speed above 100 mph**, which is physically impossible in the city.
4. **Passengers**: 117,381 trips with 0 passengers and 57 with 7-9 passengers → driver input errors.
5. **Amounts**: **7,129 negative fares** (refunds/voids that cancel a positive row), **22 fares above 1,000 $** including **623,259.86 $ for 2.4 miles** (typo), and 6,455 trips where the tip is larger than the fare (possible for a short ride, but suspicious in such numbers).

**Consequence:** `df_clean` removes these cases (3% of the rows). For **counts** (days, hours) all January trips are kept: a trip with a wrong amount is still a real pick-up.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

#### 2.1 Loading the taxi zone lookup

It is a CSV file: as recommended above, an **explicit schema** is provided instead of inferring it.

In [19]:
zones_path = download(f"{BASE_URL}/misc/taxi_zone_lookup.csv", os.path.join(DATA_DIR, "taxi_zone_lookup.csv"))

zones_schema = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .schema(zones_schema) \
    .load(zones_path)

df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


The trips are joined twice with the zone table: once on the pick-up location (`PULocationID`) and once on the drop-off location (`DOLocationID`). The zone table only has 265 rows, so `F.broadcast()` sends it to every executor and avoids shuffling the 7.7 M trips.

In [20]:
def add_boroughs(df):
    pu = df_zones.select(F.col("LocationID").alias("PULocationID"),
                         F.col("Borough").alias("PU_Borough"), F.col("Zone").alias("PU_Zone"))
    do = df_zones.select(F.col("LocationID").alias("DOLocationID"),
                         F.col("Borough").alias("DO_Borough"), F.col("Zone").alias("DO_Zone"))
    return df.join(F.broadcast(pu), "PULocationID", "left") \
             .join(F.broadcast(do), "DOLocationID", "left")

df_jan_z = add_boroughs(df_jan)
df_clean_z = add_boroughs(df_clean)

df_jan_z.select("trip_id", "PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone").show(5, truncate=False)

+-----------+----------+-----------------------------+----------+-----------------------------+
|trip_id    |PU_Borough|PU_Zone                      |DO_Borough|DO_Zone                      |
+-----------+----------+-----------------------------+----------+-----------------------------+
|25769803776|Manhattan |Manhattan Valley             |Manhattan |Upper West Side South        |
|25769803777|Manhattan |Upper West Side South        |Manhattan |West Chelsea/Hudson Yards    |
|25769803783|Manhattan |Midtown North                |Manhattan |Sutton Place/Turtle Bay North|
|25769803784|Manhattan |Sutton Place/Turtle Bay North|Queens    |Astoria                      |
|25769803785|Manhattan |Lenox Hill West              |Manhattan |Union Sq                     |
+-----------+----------+-----------------------------+----------+-----------------------------+
only showing top 5 rows


#### 2.2 Which borough had the most pick-ups? drop-offs?

In [21]:
pickups = df_jan_z.groupBy(F.col("PU_Borough").alias("Borough")).agg(F.count("*").alias("pickups"))
dropoffs = df_jan_z.groupBy(F.col("DO_Borough").alias("Borough")).agg(F.count("*").alias("dropoffs"))

pickups.join(dropoffs, "Borough", "outer") \
    .withColumn("pct_pickups", F.round(100 * F.col("pickups") / n_jan, 2)) \
    .withColumn("pct_dropoffs", F.round(100 * F.col("dropoffs") / n_jan, 2)) \
    .orderBy(F.desc("pickups")).show()

+-------------+-------+--------+-----------+------------+
|      Borough|pickups|dropoffs|pct_pickups|pct_dropoffs|
+-------------+-------+--------+-----------+------------+
|    Manhattan|6950511| 6816936|      90.31|       88.58|
|       Queens| 471113|  340914|       6.12|        4.43|
|      Unknown| 159807|  149091|       2.08|        1.94|
|     Brooklyn|  91896|  301074|       1.19|        3.91|
|        Bronx|  18056|   58068|       0.23|        0.75|
|          N/A|   3890|   16900|       0.05|        0.22|
|          EWR|    446|   10913|       0.01|        0.14|
|Staten Island|    361|    2184|        0.0|        0.03|
+-------------+-------+--------+-----------+------------+



**Answer:** **Manhattan** dominates by far, with **90.3% of pick-ups** and 88.6% of drop-offs, followed by **Queens** (6.1% of pick-ups, thanks to JFK and LaGuardia airports). Brooklyn and the Bronx get **3 times more drop-offs than pick-ups**: yellow cabs bring people there from Manhattan but find few customers on the spot (those areas are mostly served by green cabs and ride-hailing apps). "Unknown" / "N/A" are locations 264/265 (not geolocated / outside NYC).

#### 2.3 Busy / slow times by borough (pick-up borough)

In [22]:
df_jan_z.groupBy("PU_Borough").pivot("time_of_day", TOD_LABELS).count() \
    .orderBy(F.desc(TOD_LABELS[1])).show(truncate=False)

+-------------+------------------+----------------+-------------------+-----------------+
|PU_Borough   |late night (0h-6h)|morning (6h-12h)|afternoon (12h-17h)|evening (17h-24h)|
+-------------+------------------+----------------+-------------------+-----------------+
|Manhattan    |605653            |1774040         |1912221            |2658597          |
|Queens       |42903             |107769          |130161             |190280           |
|Unknown      |14093             |39866           |44971              |60877            |
|Brooklyn     |15799             |29426           |18870              |27801            |
|Bronx        |2167              |7671            |4540               |3678             |
|N/A          |760               |905             |870                |1355             |
|Staten Island|47                |150             |82                 |82               |
|EWR          |30                |108             |206                |102              |
+---------

**Answer:**
- **Manhattan** and **Queens**: the **evening** is the busiest time (Manhattan's profile is the city's profile, since it has 90% of the trips; Queens also has the afternoon airport arrivals).
- **Brooklyn**, **Bronx** and **Staten Island**: the **morning** is the busiest time — residential areas where people leave for work.
- **Late night** is the slowest time in every borough.

#### 2.4 Busiest days of the week by borough

Same remark as in 1.7: the **average** number of trips per day is used.

In [23]:
borough_daily = df_jan_z.groupBy("PU_Borough", "pickup_date", "day_num", "day_name").count()
borough_dow = borough_daily.groupBy("PU_Borough", "day_num", "day_name") \
    .agg(F.round(F.avg("count")).alias("avg_trips"))

borough_dow.groupBy("PU_Borough").pivot("day_name", DAYS).agg(F.first("avg_trips")) \
    .orderBy(F.desc("Monday")).show()

w_busy = Window.partitionBy("PU_Borough").orderBy(F.desc("avg_trips"))
borough_dow.withColumn("rank", F.row_number().over(w_busy)).filter("rank = 1") \
    .select("PU_Borough", F.col("day_name").alias("busiest_day"), "avg_trips") \
    .orderBy(F.desc("avg_trips")).show()

+-------------+--------+--------+---------+--------+--------+--------+--------+
|   PU_Borough|  Monday| Tuesday|Wednesday|Thursday|  Friday|Saturday|  Sunday|
+-------------+--------+--------+---------+--------+--------+--------+--------+
|    Manhattan|201858.0|217239.0| 228953.0|245903.0|246224.0|231875.0|192553.0|
|       Queens| 16662.0| 15737.0|  15163.0| 15792.0| 16038.0| 11915.0| 14798.0|
|      Unknown|  5354.0|  4905.0|   5156.0|  5785.0|  5441.0|  5176.0|  4174.0|
|     Brooklyn|  2377.0|  3156.0|   3020.0|  3143.0|  3273.0|  2901.0|  2775.0|
|        Bronx|   543.0|   612.0|    600.0|   624.0|   667.0|   482.0|   528.0|
|          N/A|   129.0|   141.0|    127.0|   127.0|   111.0|   123.0|   117.0|
|Staten Island|    11.0|    11.0|     10.0|    12.0|    16.0|    10.0|    11.0|
|          EWR|     8.0|    15.0|     17.0|    12.0|    19.0|    14.0|    17.0|
+-------------+--------+--------+---------+--------+--------+--------+--------+



+-------------+-----------+---------+
|   PU_Borough|busiest_day|avg_trips|
+-------------+-----------+---------+
|    Manhattan|     Friday| 246224.0|
|       Queens|     Monday|  16662.0|
|      Unknown|   Thursday|   5785.0|
|     Brooklyn|     Friday|   3273.0|
|        Bronx|     Friday|    667.0|
|          N/A|    Tuesday|    141.0|
|          EWR|     Friday|     19.0|
|Staten Island|     Friday|     16.0|
+-------------+-----------+---------+



**Answer:**
- **Manhattan**: **Friday** is the busiest (then Thursday), Sunday the slowest.
- **Queens**: **Monday** is the busiest and Saturday the slowest (business travel through the airports).
- **Brooklyn**, **Bronx**: **Friday** is the busiest.
- EWR and Staten Island have too few trips (≈ 10-20 per day) for their ranking to be meaningful.

#### 2.5 / 2.6 Average trip distance and average fare by borough (cleaned data, pick-up borough)

In [24]:
df_clean_z.groupBy("PU_Borough").agg(
        F.count("*").alias("trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")) \
    .orderBy(F.desc("trips")).show()

+-------------+-------+---------------+--------+
|   PU_Borough|  trips|avg_distance_mi|avg_fare|
+-------------+-------+---------------+--------+
|    Manhattan|6774627|           2.24|   10.63|
|       Queens| 443730|           11.7|   35.81|
|      Unknown| 141352|           2.58|   11.57|
|     Brooklyn|  87036|           4.96|   18.97|
|        Bronx|  16653|           7.74|   27.33|
|          N/A|   1606|           5.03|   32.11|
|Staten Island|    296|          14.94|   47.84|
|          EWR|     35|          10.01|   63.54|
+-------------+-------+---------------+--------+



**Answer:**
- **Manhattan**: short trips, 2.24 miles and **10.63 $** on average.
- **Queens**: 11.7 miles and **35.81 $** (airports; JFK → Manhattan is a 52 $ flat fare in 2019).
- **Staten Island**: longest average distance (14.94 miles, 47.84 $) but only 296 trips.
- **EWR (Newark)**: highest average fare (**63.54 $**), only 35 trips (outside NYC: metered fare + surcharges).
- **Bronx**: 7.74 miles / 27.33 $; **Brooklyn**: 4.96 miles / 18.97 $.

#### 2.7 Highest / lowest fare amounts and associated borough

In [25]:
fare_cols = ["trip_id", "trip_distance", "duration_min", "fare_amount",
             "PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone"]

print("=== Highest fare (raw) ===")
df_jan_z.orderBy(F.desc("fare_amount")).select(fare_cols).show(3, truncate=False)
print("=== Lowest fare (raw) ===")
df_jan_z.orderBy("fare_amount").select(fare_cols).show(3, truncate=False)

print("=== Highest fare (cleaned) ===")
df_clean_z.orderBy(F.desc("fare_amount")).select(fare_cols).show(3, truncate=False)
print("=== Lowest fare (cleaned) ===")
df_clean_z.orderBy("fare_amount", "trip_distance").select(fare_cols).show(3, truncate=False)

=== Highest fare (raw) ===


+-----------+-------------+------------+-----------+----------+---------------------+----------+--------+
|trip_id    |trip_distance|duration_min|fare_amount|PU_Borough|PU_Zone              |DO_Borough|DO_Zone |
+-----------+-------------+------------+-----------+----------+---------------------+----------+--------+
|25772303431|2.4          |19.9        |623259.86  |Manhattan |Upper East Side South|Manhattan |Flatiron|
|25775127259|0.0          |0.0         |355676.98  |Manhattan |Bloomingdale         |Unknown   |N/A     |
|25771963747|0.0          |0.0         |36090.3    |Unknown   |N/A                  |Unknown   |N/A     |
+-----------+-------------+------------+-----------+----------+---------------------+----------+--------+
only showing top 3 rows
=== Lowest fare (raw) ===


+-----------+-------------+-------------------+-----------+----------+---------------------------------+----------+---------------------------------+
|trip_id    |trip_distance|duration_min       |fare_amount|PU_Borough|PU_Zone                          |DO_Borough|DO_Zone                          |
+-----------+-------------+-------------------+-----------+----------+---------------------------------+----------+---------------------------------+
|25774694425|0.0          |6.85               |-362.0     |Queens    |JFK Airport                      |Queens    |JFK Airport                      |
|25776111976|0.0          |0.13333333333333333|-320.0     |N/A       |Outside of NYC                   |N/A       |Outside of NYC                   |
|25769860870|0.0          |4.183333333333334  |-300.0     |Bronx     |University Heights/Morris Heights|Bronx     |University Heights/Morris Heights|
+-----------+-------------+-------------------+-----------+----------+------------------------------

+-----------+-------------+------------------+-----------+----------+--------------+----------+--------------+
|trip_id    |trip_distance|duration_min      |fare_amount|PU_Borough|PU_Zone       |DO_Borough|DO_Zone       |
+-----------+-------------+------------------+-----------+----------+--------------+----------+--------------+
|25770841720|1.5          |1.0333333333333334|500.0      |Queens    |Middle Village|Queens    |Middle Village|
|25775340491|0.1          |1.2833333333333334|500.0      |N/A       |Outside of NYC|N/A       |Outside of NYC|
|25770080355|74.24        |114.55            |468.0      |Queens    |JFK Airport   |N/A       |Outside of NYC|
+-----------+-------------+------------------+-----------+----------+--------------+----------+--------------+
only showing top 3 rows
=== Lowest fare (cleaned) ===


+-----------+-------------+-----------------+-----------+----------+------------+----------+-------------------------+
|trip_id    |trip_distance|duration_min     |fare_amount|PU_Borough|PU_Zone     |DO_Borough|DO_Zone                  |
+-----------+-------------+-----------------+-----------+----------+------------+----------+-------------------------+
|25769825313|0.1          |3.533333333333333|0.01       |Manhattan |Clinton East|Manhattan |Clinton East             |
|25769836762|0.1          |3.066666666666667|0.01       |Manhattan |Gramercy    |Manhattan |Gramercy                 |
|25769821147|0.1          |6.983333333333333|0.01       |Manhattan |East Chelsea|Manhattan |West Chelsea/Hudson Yards|
+-----------+-------------+-----------------+-----------+----------+------------+----------+-------------------------+
only showing top 3 rows


**Answer:**
- **Highest fare (raw): 623,259.86 $**, trip `25772303431`, from **Manhattan** (Upper East Side South) to Manhattan (Flatiron), 2.4 miles in 20 min → obvious typo (such a trip costs about 12 $).
- **Lowest fare (raw): −362 $**, trip from **Queens** (JFK Airport) to JFK, 0 miles → refund / void.
- **On the cleaned data** the maximum (500 $) is the filter threshold and still suspicious (1.5 miles in 1 min in Queens); the most credible expensive trip is **468 $ from JFK (Queens) to outside NYC**, 74 miles in almost 2 h. The minimum is **0.01 $** in Manhattan (Clinton East) on New Year's night, also suspicious.

#### 2.8 Comparison with January 2025

January 2025 is loaded and processed with **exactly** the same `enrich`, `january_only` and `clean` functions.

Note: the schema changed between 2019 and 2025 (`airport_fee` → `Airport_fee`, new `cbd_congestion_fee` column for the Manhattan congestion pricing started on January 5, 2025, `passenger_count` stored as `long` instead of `double`). Only common columns are used here, so this is not a problem.

In [26]:
df_jan_2025 = january_only(enrich(spark.read.parquet(download_trips(2025, 1))), 2025)

def summarize(df, year):
    cleaned = clean(df)
    return cleaned.agg(
        F.lit(str(year)).alias("year"),
        F.count("*").alias("trips_clean"),
        F.round(F.avg("passenger_count"), 3).alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("total_amount"), 2).alias("avg_total"))

print("January 2019 trips:", f"{df_jan.count():,}")
print("January 2025 trips:", f"{df_jan_2025.count():,}")

# one row per metric, one column per year
comparison = summarize(df_jan, 2019).unionByName(summarize(df_jan_2025, 2025)).toPandas().set_index("year").T
comparison["change_%"] = ((comparison["2025"] / comparison["2019"] - 1) * 100).round(1)
comparison

January 2019 trips: 7,696,080


January 2025 trips: 3,475,204


year,2019,2025,change_%
trips_clean,7465335.000,3207764.000,-57.0
avg_passengers,1.592,1.306,-18.0
avg_distance_mi,2.860,3.190,11.5
avg_duration_min,13.040,14.730,13.0
avg_fare,12.290,17.930,45.9
avg_tip,1.810,3.130,72.9
avg_total,15.560,26.810,72.3


**Answer — January 2019 vs January 2025:**

- **Number of trips: more than halved** (7.7 M → 3.5 M): ride-hailing apps (Uber, Lyft) took a large share of the market, and COVID changed commuting habits.
- **Average passengers**: 1.59 → 1.31.
- **Average distance +12%** (2.86 → 3.19 miles) and **average duration +13%** (13.0 → 14.7 min).
- **Average fare +46%** (12.29 → 17.93 $): fare increase at the end of 2022 (+23% on the base fare and per-mile rate) and inflation.
- **Average total paid +72%** (15.56 → 26.81 $), growing faster than the fare because of the new surcharges (`congestion_surcharge` of 2.50 $ since February 2019, `cbd_congestion_fee` since January 2025, `Airport_fee`).
- **Average tip +73%** (1.81 → 3.13 $): payment screens suggest higher tip percentages.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

---

The main lab was done in PySpark, so **3 questions are redone in pure Spark SQL** (the last one with a join). The DataFrames are first registered as temporary views.

In [27]:
df_trips.createOrReplaceTempView("trips")          # raw
df_jan.createOrReplaceTempView("trips_jan")        # January 2019
df_zones.createOrReplaceTempView("zones")

#### SQL 1 — Highest and average passenger count (questions 1.2 and 1.3)

In [28]:
spark.sql('''
    WITH m AS (SELECT MAX(passenger_count) AS max_passengers FROM trips)
    SELECT m.max_passengers,
           SUM(CASE WHEN t.passenger_count = m.max_passengers THEN 1 ELSE 0 END) AS trips_with_max,
           ROUND(AVG(t.passenger_count), 3)                                      AS avg_passengers
    FROM trips t
    CROSS JOIN m
    GROUP BY m.max_passengers
''').show()

+--------------+--------------+--------------+
|max_passengers|trips_with_max|avg_passengers|
+--------------+--------------+--------------+
|           9.0|             9|         1.567|
+--------------+--------------+--------------+



#### SQL 2 — Busiest / slowest day of the week on average (question 1.7)

In [29]:
spark.sql('''
    WITH daily AS (
        SELECT pickup_date, day_num, day_name, COUNT(*) AS trips
        FROM trips_jan
        GROUP BY pickup_date, day_num, day_name
    )
    SELECT day_num, day_name,
           COUNT(*)             AS nb_days,
           ROUND(AVG(trips))    AS avg_trips_per_day
    FROM daily
    GROUP BY day_num, day_name
    ORDER BY day_num
''').show()

+-------+---------+-------+-----------------+
|day_num| day_name|nb_days|avg_trips_per_day|
+-------+---------+-------+-----------------+
|      1|   Monday|      4|         226941.0|
|      2|  Tuesday|      5|         241815.0|
|      3|Wednesday|      5|         253046.0|
|      4| Thursday|      5|         271398.0|
|      5|   Friday|      4|         271788.0|
|      6| Saturday|      4|         252495.0|
|      7|   Sunday|      4|         214973.0|
+-------+---------+-------+-----------------+



#### SQL 3 — Pick-ups and drop-offs by borough (question 2.2, **join**)

In [30]:
spark.sql('''
    WITH pickups AS (
        SELECT z.Borough, COUNT(*) AS pickups
        FROM trips_jan t
        LEFT JOIN zones z ON t.PULocationID = z.LocationID
        GROUP BY z.Borough
    ),
    dropoffs AS (
        SELECT z.Borough, COUNT(*) AS dropoffs
        FROM trips_jan t
        LEFT JOIN zones z ON t.DOLocationID = z.LocationID
        GROUP BY z.Borough
    )
    SELECT COALESCE(p.Borough, d.Borough) AS Borough, p.pickups, d.dropoffs
    FROM pickups p
    FULL OUTER JOIN dropoffs d ON p.Borough = d.Borough
    ORDER BY p.pickups DESC
''').show()

+-------------+-------+--------+
|      Borough|pickups|dropoffs|
+-------------+-------+--------+
|    Manhattan|6950511| 6816936|
|       Queens| 471113|  340914|
|      Unknown| 159807|  149091|
|     Brooklyn|  91896|  301074|
|        Bronx|  18056|   58068|
|          N/A|   3890|   16900|
|          EWR|    446|   10913|
|Staten Island|    361|    2184|
+-------------+-------+--------+



The SQL results are identical to the PySpark ones: both APIs go through the same **Catalyst optimizer**.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

#### Full year 2019 and busiest season

The 12 monthly files are downloaded (≈ 1.3 GB, only the first time). Some column types change from one month to another, so each file is read separately keeping only the pick-up datetime, then combined with `unionByName`.

Meteorological seasons: winter = Dec/Jan/Feb, spring = Mar-May, summer = Jun-Aug, fall = Sep-Nov.

In [31]:
df_2019 = None
for m in range(1, 13):
    df_month = spark.read.parquet(download_trips(2019, m)).select("tpep_pickup_datetime")
    df_2019 = df_month if df_2019 is None else df_2019.unionByName(df_month)

df_2019 = df_2019.withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .filter(F.year("pickup_date") == 2019)

month = F.month("pickup_date")
season = (F.when(month.isin(12, 1, 2), "winter")
           .when(month.isin(3, 4, 5), "spring")
           .when(month.isin(6, 7, 8), "summer")
           .otherwise("fall"))

monthly_2019 = df_2019.groupBy(month.alias("month")).count().orderBy("month")

df_2019.withColumn("season", season).groupBy("season").agg(
        F.count("*").alias("trips"),
        F.countDistinct("pickup_date").alias("nb_days")) \
    .withColumn("avg_trips_per_day", F.round(F.col("trips") / F.col("nb_days"))) \
    .orderBy(F.desc("avg_trips_per_day")).show()

+------+--------+-------+-----------------+
|season|   trips|nb_days|avg_trips_per_day|
+------+--------+-------+-----------------+
|spring|22940902|     92|         249358.0|
|winter|21641751|     90|         240464.0|
|  fall|20659549|     91|         227028.0|
|summer|19354800|     92|         210378.0|
+------+--------+-------+-----------------+



**Answer:** measured in average trips per day, **spring is the busiest season** (249,358 trips/day), ahead of winter (240,464), fall (227,028) and **summer, the slowest** (210,378). Busiest month: **March** (7.87 M trips); slowest: **August** (6.07 M).

#### Native Spark 4 visualizations (`DataFrame.plot`)

Since Spark 4, `df.plot.<kind>(...)` draws a **Plotly** chart directly from a Spark DataFrame (it is converted to pandas behind the scenes, so the data must be aggregated first). Plotly is not included in the Docker image, so it is installed here (to be redone each time the container is restarted, since it runs with `--rm`).

In [32]:
%pip install -q plotly

Note: you may need to restart the kernel to use updated packages.


In [33]:
fig = hourly.plot.line(x="pickup_hour", y="count", markers=True)
fig.update_layout(title="January 2019 — number of trips by pick-up hour (question 1.6)",
                  xaxis_title="Hour", yaxis_title="Trips")
fig.show()

In [34]:
fig = dow.plot.bar(x="day_name", y="avg_trips_per_day")
fig.update_layout(title="January 2019 — average number of trips by day of the week (question 1.7)",
                  xaxis_title="", yaxis_title="Trips / day")
fig.show()

In [35]:
fig = monthly_2019.plot.bar(x="month", y="count")
fig.update_layout(title="2019 — number of trips by month (busiest season)",
                  xaxis_title="Month", yaxis_title="Trips")
fig.show()